# Data Cleaning, Preprocessing & ML — Timed-Assessment Drill

**Target: BCG Forward Deployed AI Scientist — CodeSignal (Sat).**
Skills tested: *Data cleaning & preprocessing · Querying & retrieval · EDA & visualization · ML & predictive modeling.*

**How to use this notebook (ARENA-style):**
1. Read the exercise. There's a difficulty/time/skill tag.
2. Fill in the code cell. The blanks are marked `# TODO` / `...`.
3. Run the cell. Only then expand the **Solution** to compare.
4. The goal is **fingers-from-memory under a clock**, not reading. If you peek before trying, you wasted the rep.

**The leak rule (memorize):** fit imputers / scalers / encoders on **train only**, then transform test. Violating this is the #1 thing graders catch. The `Pipeline` section at the end makes this automatic.

No Kaggle key needed — we use the built-in seaborn Titanic dataset (a classic binary-classification cleaning problem). Swap in a Kaggle CSV for the ML section later if you want.


## 0 — Setup

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)
plt.rcParams["figure.figsize"] = (7, 4)

# Titanic: a deliberately messy classic. target = `survived` (0/1).
df = sns.load_dataset("titanic")
print(df.shape)
df.head()

(891, 15)


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


---
# Part 1 — Reconnaissance (the first 5 minutes of any assessment)

Before touching anything, you build a mental model of the data. Do this *fast and the same way every time*. The muscle memory here buys you time later.

### Exercise - First-look snapshot

> ```yaml
> Difficulty: 🔴⚪⚪⚪⚪
> Time: 3 min
> Skill: Data cleaning & preprocessing
> ```

Write the four-line reconnaissance you run on **every** new dataframe:
* shape
* `info()` (dtypes + non-null counts — your first missingness read)
* `describe()` over **all** columns (not just numeric)
* count of missing values per column, sorted descending


In [3]:
# TODO: shape
print("type:", type(df))
print("shape:", df.shape)
# TODO: dtypes + non-null counts
display(df.describe(include="all").T)
# TODO: summary stats over ALL columns (numeric + categorical)
print(df.info())
# TODO: missing values per column, descending
df.isna().sum().sort_values(ascending=False).T

type: <class 'pandas.DataFrame'>
shape: (891, 15)


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
survived,891.0,NaN,NaN,NaN,0.383838,0.486592,0.0,0.0,0.0,1.0,1.0
pclass,891.0,NaN,NaN,NaN,2.308642,0.836071,1.0,2.0,3.0,3.0,3.0
sex,891,2,male,577,NaN,NaN,NaN,NaN,NaN,NaN,NaN
age,714.0,NaN,NaN,NaN,29.699118,14.526497,0.42,20.125,28.0,38.0,80.0
sibsp,891.0,NaN,NaN,NaN,0.523008,1.102743,0.0,0.0,0.0,1.0,8.0
parch,891.0,NaN,NaN,NaN,0.381594,0.806057,0.0,0.0,0.0,0.0,6.0
fare,891.0,NaN,NaN,NaN,32.204208,49.693429,0.0,7.9104,14.4542,31.0,512.3292
embarked,889,3,S,644,NaN,NaN,NaN,NaN,NaN,NaN,NaN
class,891,3,Third,491,NaN,NaN,NaN,NaN,NaN,NaN,NaN
who,891,3,man,537,NaN,NaN,NaN,NaN,NaN,NaN,NaN


<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    str     
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    str     
 8   class        891 non-null    category
 9   who          891 non-null    str     
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    str     
 13  alive        891 non-null    str     
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), str(5)
memory usage: 80.7 KB
None


deck           688
age            177
embarked         2
embark_town      2
sex              0
pclass           0
survived         0
fare             0
parch            0
sibsp            0
class            0
adult_male       0
who              0
alive            0
alone            0
dtype: int64

<details><summary>Solution</summary>

```python
print(df.shape)
df.info()
display(df.describe(include="all").T)
df.isna().sum().sort_values(ascending=False)
```

</details>

### Exercise - Cardinality & dtype audit

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Time: 4 min
> Skill: Data cleaning & preprocessing
> ```

For each column print its dtype and number of unique values. This tells you:
* which "numeric" columns are really categorical codes,
* which object columns are high-cardinality (one-hot will explode),
* candidate ID columns to drop.

Build a small summary dataframe with columns: `dtype`, `n_unique`, `n_missing`, `pct_missing`.


In [4]:
(df.isna().sum()/df.shape[0] * 100).round(1)
(df.isna() / df.shape[0] * 100).round(2)


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.11,0.0,0.0,0.0
1,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0
2,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.11,0.0,0.0,0.0
3,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0
4,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.11,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.11,0.0,0.0,0.0
887,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0
888,0.0,0.0,0.0,0.11,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.11,0.0,0.0,0.0
889,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0


In [5]:
summary = pd.DataFrame({
    "dtype": df.dtypes,
    "n_unique": df.nunique(),
    "n_missing": df.isna().sum(),
})
summary["pct_missing"] = (df.isna().sum() / df.shape[0] * 100).round(2)
summary.sort_values("pct_missing", ascending=False)


summary = pd.DataFrame({
    "dtype": df.dtypes,
    "n_unique": df.nunique(),
    "n_missing": df.isna().sum(),
    "pct_missig": (df.isna().sum() / df.shape[0] * 100).round(2)
})

summary.sort_values("n_missing", ascending=False)

,dtype,n_unique,n_missing,pct_missig
deck,category,7,688,77.22
age,float64,88,177,19.87
embarked,str,3,2,0.22
embark_town,str,3,2,0.22
sex,str,2,0,0.00
pclass,int64,3,0,0.00
survived,int64,2,0,0.00
fare,float64,248,0,0.00
parch,int64,7,0,0.00
sibsp,int64,7,0,0.00


<details><summary>Solution</summary>

```python
summary = pd.DataFrame({
    "dtype": df.dtypes,
    "n_unique": df.nunique(),
    "n_missing": df.isna().sum(),
})
summary["pct_missing"] = (summary["n_missing"] / len(df) * 100).round(1)
summary.sort_values("pct_missing", ascending=False)
```

</details>

---
# Part 2 — Missing data (the heart of cleaning)

Three questions, always in this order:
1. **How much** is missing, and where? (Part 1 answered this.)
2. **Why** is it missing — MCAR / MAR / MNAR? Missingness can itself be predictive.
3. **What** do I do — drop, impute, or flag?

Rules of thumb: a column that's >50–60% missing is usually droppable; a row missing the *target* is droppable; everything else gets imputed **with the train statistic** (median for skewed numerics, mode/`"Missing"` for categoricals).

### Exercise - Missingness as a signal

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Time: 5 min
> Skill: Data cleaning & preprocessing
> ```

Sometimes *whether* a value is missing predicts the target better than the value itself.

For `deck` (lots of missing), create a binary indicator `deck_missing` and compare the **survival rate** for missing vs non-missing. If the rates differ a lot, you keep the indicator as a feature instead of just imputing it away.


In [6]:
df["deck_missing"] = df["deck"].isna().astype(int)
# df["deck_missing"] = df["deck"].isna()
# survival rate grouped by the indicator
a = df.groupby("deck_missing")["survived"].mean()
a

deck_missing
0    0.669951
1    0.299419
Name: survived, dtype: float64

<details><summary>Solution</summary>

```python
df["deck_missing"] = df["deck"].isna().astype(int)
df.groupby("deck_missing")["survived"].mean()
```

</details>

### Exercise - Group-wise imputation (better than a global median)

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Time: 6 min
> Skill: Data cleaning & preprocessing
> ```

`age` is missing ~20%. A global median ignores structure. Passenger class and sex are strongly tied to age, so impute the **median age within each (`pclass`, `sex`) group**.

Use `groupby(...).transform(...)` so the result aligns back to the original index. Fill `age` with it.


In [ ]:
df["age"] = df["age"].fillna(
    df.groupby(["pclass", "sex"])["age"].transform("mean")
)
df["age"].isna().sum()

np.int64(0)

<details><summary>Solution</summary>

```python
df["age"] = df["age"].fillna(
    df.groupby(["pclass", "sex"])["age"].transform("median")
)
df["age"].isna().sum()
```

</details>

### Exercise - Simple categorical imputation + drop a dead column

> ```yaml
> Difficulty: 🔴⚪⚪⚪⚪
> Time: 3 min
> Skill: Data cleaning & preprocessing
> ```

* `embarked` / `embark_town` have a couple missing — fill with the mode.
* `deck` is >70% missing — we already captured `deck_missing`, so drop the raw `deck` column.


In [12]:
df["embarked"] = df["embarked"].fillna(df["embarked"].mode)
df["embark_town"] = df["embark_town"].fillna(df["embark_town"])
df = df.drop("deck", axis=1)
df.isna().sum().sort_values(ascending=False)



# df["embarked"] = df["embarked"].fillna(...)
# df["embark_town"] = df["embark_town"].fillna(...)
# df = df.drop(columns=[...])
# df.isna().sum().sort_values(ascending=False).head()

survived        0
pclass          0
sex             0
age             0
sibsp           0
parch           0
fare            0
embarked        0
class           0
who             0
adult_male      0
embark_town     0
alive           0
alone           0
deck_missing    0
dtype: int64

<details><summary>Solution</summary>

```python
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
df["embark_town"] = df["embark_town"].fillna(df["embark_town"].mode()[0])
df = df.drop(columns=["deck"])
df.isna().sum().sort_values(ascending=False).head()
```

</details>

---
# Part 3 — Dtypes, duplicates, and outliers

### Exercise - Coerce dtypes & convert to category

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Time: 4 min
> Skill: Data cleaning & preprocessing
> ```

`pclass` is an integer but is really an **ordinal category**. `sex`, `embarked`, `class`, `who`, `alive` are categoricals stored as objects/bools.

Convert `pclass` to an ordered categorical (1 < 2 < 3) and cast the obvious object columns to `category` dtype (smaller memory, plays nice with groupby). Note `alive` is a leak of the target — flag it mentally to drop before modeling.


In [ ]:
df["pclass"] = pd.Categorical(df["pclass"], categories=[1, 2, 3], ordered=True)
for col in [...]:
    df[col] = df[col].astype("category")
df.dtypes

<details><summary>Solution</summary>

```python
df["pclass"] = pd.Categorical(df["pclass"], categories=[1, 2, 3], ordered=True)
for col in ["sex", "embarked", "embark_town", "class", "who"]:
    df[col] = df[col].astype("category")
df.dtypes
# NB: `alive` is literally survived in words -> target leak. Drop before modeling.
```

</details>

### Exercise - Duplicate rows

> ```yaml
> Difficulty: 🔴⚪⚪⚪⚪
> Time: 2 min
> Skill: Data cleaning & preprocessing
> ```

Count fully-duplicated rows, then drop them (keeping the first). Report shape before and after.


In [ ]:
before = df.shape
# TODO count dupes
...
# TODO drop
...
print(before, "->", df.shape)

<details><summary>Solution</summary>

```python
before = df.shape
print("dupes:", df.duplicated().sum())
df = df.drop_duplicates(keep="first")
print(before, "->", df.shape)
```

</details>

### Exercise - Outlier detection with the IQR rule

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Time: 6 min
> Skill: Data cleaning & preprocessing
> ```

Write a reusable function `iqr_bounds(s, k=1.5)` returning the lower/upper fences (Q1 − k·IQR, Q3 + k·IQR). Apply it to `fare`, report how many points fall outside, then **winsorize** (clip) `fare` to those bounds rather than deleting rows — clipping keeps the sample size.


In [ ]:
def iqr_bounds(s, k=1.5):
    q1, q3 = ...
    iqr = ...
    return ..., ...

lo, hi = iqr_bounds(df["fare"])
print("outliers:", ((df["fare"] < lo) | (df["fare"] > hi)).sum())
df["fare"] = ...  # clip to [lo, hi]

<details><summary>Solution</summary>

```python
def iqr_bounds(s, k=1.5):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

lo, hi = iqr_bounds(df["fare"])
print("outliers:", ((df["fare"] < lo) | (df["fare"] > hi)).sum())
df["fare"] = df["fare"].clip(lower=lo, upper=hi)
```

</details>

---
# Part 4 — Feature engineering

Cheap, high-signal transforms graders love to see: binning, extracting structure, and combining columns into a more informative one.

### Exercise - Binning a continuous variable

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Time: 4 min
> Skill: Data cleaning & preprocessing
> ```

Create `age_band` by cutting `age` into labelled bins: child (0–12), teen (12–18), adult (18–60), senior (60+). Use `pd.cut`. Then show survival rate per band.


In [ ]:
df["age_band"] = pd.cut(
    df["age"],
    bins=[...],
    labels=[...],
)
df.groupby("age_band", observed=True)["survived"].mean()

<details><summary>Solution</summary>

```python
df["age_band"] = pd.cut(
    df["age"],
    bins=[0, 12, 18, 60, 200],
    labels=["child", "teen", "adult", "senior"],
)
df.groupby("age_band", observed=True)["survived"].mean()
```

</details>

### Exercise - Combine columns into a feature

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Time: 4 min
> Skill: Data cleaning & preprocessing
> ```

`sibsp` (siblings/spouses) + `parch` (parents/children) + self = family size. Build `family_size`, then a boolean `is_alone` (family_size == 1). Compare survival of alone vs not.


In [ ]:
df["family_size"] = ...
df["is_alone"] = ...
df.groupby("is_alone")["survived"].mean()

<details><summary>Solution</summary>

```python
df["family_size"] = df["sibsp"] + df["parch"] + 1
df["is_alone"] = (df["family_size"] == 1).astype(int)
df.groupby("is_alone")["survived"].mean()
```

</details>

---
# Part 5 — Querying & retrieval (pandas as SQL)

CodeSignal often hides a "compute this aggregate" question. Speed on `groupby`/`merge`/`pivot` is free points.

### Exercise - groupby + multi-aggregate

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Time: 5 min
> Skill: Querying & retrieval
> ```

For each (`pclass`, `sex`) produce: count of passengers, mean fare, and survival rate — in one `groupby().agg()` call with named outputs.


In [ ]:
df.groupby([...], observed=True).agg(
    n=(..., ...),
    mean_fare=(..., ...),
    survival_rate=(..., ...),
).round(2)

<details><summary>Solution</summary>

```python
df.groupby(["pclass", "sex"], observed=True).agg(
    n=("survived", "size"),
    mean_fare=("fare", "mean"),
    survival_rate=("survived", "mean"),
).round(2)
```

</details>

### Exercise - pivot_table + transform vs agg

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Time: 5 min
> Skill: Querying & retrieval
> ```

(a) Build a `pivot_table` of survival rate with `pclass` as rows and `sex` as columns.
(b) Add a column `fare_vs_class_mean` = each passenger's fare minus the mean fare of their `pclass`. This needs `transform` (broadcasts back to row level), not `agg`.


In [ ]:
pivot = pd.pivot_table(df, values=..., index=..., columns=..., aggfunc=...)
display(pivot.round(2))

df["fare_vs_class_mean"] = df["fare"] - df.groupby("pclass", observed=True)["fare"].transform(...)
df[["pclass", "fare", "fare_vs_class_mean"]].head()

<details><summary>Solution</summary>

```python
pivot = pd.pivot_table(df, values="survived", index="pclass", columns="sex", aggfunc="mean")
display(pivot.round(2))

df["fare_vs_class_mean"] = df["fare"] - df.groupby("pclass", observed=True)["fare"].transform("mean")
df[["pclass", "fare", "fare_vs_class_mean"]].head()
```

</details>

---
# Part 6 — EDA & visualization

The point is **speed of insight**, not pretty charts. Have a small visual vocabulary you can type blind: distribution, box-by-category, correlation heatmap, target-vs-feature.

### Exercise - Distribution + skew

> ```yaml
> Difficulty: 🔴⚪⚪⚪⚪
> Time: 3 min
> Skill: EDA & visualization
> ```

Plot a histogram of `fare` and print its skew. (Right-skewed money variables often want a log transform — note it if skew > 1.)


In [ ]:
print("skew:", df["fare"].skew().round(2))
...  # histogram

<details><summary>Solution</summary>

```python
print("skew:", df["fare"].skew().round(2))
df["fare"].hist(bins=40)
plt.title("fare distribution"); plt.xlabel("fare"); plt.show()
```

</details>

### Exercise - Correlation heatmap

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Time: 4 min
> Skill: EDA & visualization
> ```

Compute the correlation matrix of numeric columns and draw a `seaborn` heatmap with annotations. Look for multicollinearity (|r| > 0.8) and features correlated with `survived`.


In [ ]:
num = df.select_dtypes(include=...)
corr = ...
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.show()

<details><summary>Solution</summary>

```python
num = df.select_dtypes(include="number")
corr = num.corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.show()
```

</details>

### Exercise - Target-vs-feature (the money plot)

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Time: 4 min
> Skill: EDA & visualization
> ```

Two quick reads of how features relate to the target:
* a `crosstab` of `pclass` × `survived` normalized by row (survival rate per class),
* a `sns.barplot` of survival by `sex`.


In [ ]:
display(pd.crosstab(df["pclass"], df["survived"], normalize=...))
sns.barplot(data=df, x=..., y=...)
plt.show()

<details><summary>Solution</summary>

```python
display(pd.crosstab(df["pclass"], df["survived"], normalize="index"))
sns.barplot(data=df, x="sex", y="survived")
plt.show()
```

</details>

---
# Part 7 — The leak-proof Pipeline (★ the centerpiece ★)

This is the single highest-leverage thing to know from memory. A `ColumnTransformer` + `Pipeline` bundles imputation + scaling + encoding + model into **one object** that `fit`s on train only and `transform`s test automatically — no leakage, no hand-wiring. **Type this from a blank cell every day until Friday.**

### Exercise - Build the ColumnTransformer + Pipeline from blank

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Time: 10 min
> Skill: ML & predictive modeling
> ```

Starting from the raw seaborn Titanic frame again (clean slate), build the full preprocessing+model pipeline:
* numeric branch: median impute → standard scale
* categorical branch: most-frequent impute → one-hot (`handle_unknown="ignore"`)
* combine with `ColumnTransformer`
* attach a `RandomForestClassifier`

Drop leak/duplicate columns: `alive`, `who`, `adult_male`, `class`, `embark_town`, `deck`. Then `train_test_split` with **stratify** and fit.


In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

data = sns.load_dataset("titanic").drop(
    columns=["alive", "who", "adult_male", "class", "embark_town", "deck"]
)
y = data.pop("survived")
X = data

num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = X.select_dtypes(exclude="number").columns.tolist()

num = Pipeline([...])
cat = Pipeline([...])
pre = ColumnTransformer([...])
model = Pipeline([("pre", pre), ("clf", RandomForestClassifier(random_state=0))])

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=..., random_state=0)
model.fit(X_tr, y_tr)
print("test acc:", round(model.score(X_te, y_te), 3))

<details><summary>Solution</summary>

```python
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

data = sns.load_dataset("titanic").drop(
    columns=["alive", "who", "adult_male", "class", "embark_town", "deck"]
)
y = data.pop("survived")
X = data

num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = X.select_dtypes(exclude="number").columns.tolist()

num = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("sc", StandardScaler()),
])
cat = Pipeline([
    ("imp", SimpleImputer(strategy="most_frequent")),
    ("oh", OneHotEncoder(handle_unknown="ignore")),
])
pre = ColumnTransformer([
    ("num", num, num_cols),
    ("cat", cat, cat_cols),
])
model = Pipeline([("pre", pre), ("clf", RandomForestClassifier(random_state=0))])

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
model.fit(X_tr, y_tr)
print("test acc:", round(model.score(X_te, y_te), 3))
```

</details>

---
# Part 8 — ML & evaluation

### Exercise - Baseline first, then cross-validate

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Time: 5 min
> Skill: ML & predictive modeling
> ```

Always state a baseline before celebrating a model. (a) Print the majority-class baseline accuracy. (b) Run 5-fold `cross_val_score` on the pipeline with `roc_auc` and report mean ± std.


In [ ]:
baseline = ...  # majority-class accuracy
print("baseline acc:", round(baseline, 3))

scores = cross_val_score(model, X, y, cv=..., scoring=...)
print(f"CV ROC-AUC: {scores.mean():.3f} +/- {scores.std():.3f}")

<details><summary>Solution</summary>

```python
baseline = y.value_counts(normalize=True).max()
print("baseline acc:", round(baseline, 3))

scores = cross_val_score(model, X, y, cv=5, scoring="roc_auc")
print(f"CV ROC-AUC: {scores.mean():.3f} +/- {scores.std():.3f}")
```

</details>

### Exercise - Classification report + confusion matrix

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Time: 5 min
> Skill: ML & predictive modeling
> ```

On the held-out test set: print `classification_report` (precision/recall/F1) and the confusion matrix. Know what each metric means — graders ask "why F1 not accuracy?" (imbalanced classes).


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
y_pred = ...
print(classification_report(y_te, y_pred))
print(confusion_matrix(y_te, y_pred))

<details><summary>Solution</summary>

```python
from sklearn.metrics import classification_report, confusion_matrix
y_pred = model.predict(X_te)
print(classification_report(y_te, y_pred))
print(confusion_matrix(y_te, y_pred))
```

</details>

### Exercise - Feature importance (with one-hot names)

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Time: 6 min
> Skill: ML & predictive modeling
> ```

Pull feature names out of the fitted `ColumnTransformer` (`get_feature_names_out`) and pair them with the random forest's `feature_importances_`. Show the top 10. This is the "what drove the prediction" question that always comes up.


In [ ]:
importances = model.named_steps["clf"].feature_importances_
names = model.named_steps["pre"].get_feature_names_out()
imp = pd.Series(..., index=...).sort_values(ascending=False)
imp.head(10)

<details><summary>Solution</summary>

```python
importances = model.named_steps["clf"].feature_importances_
names = model.named_steps["pre"].get_feature_names_out()
imp = pd.Series(importances, index=names).sort_values(ascending=False)
imp.head(10)
```

</details>

---
# Part 9 — Timed-assessment cheat sheet (read Friday night, don't memorize cold)

**The 6-step flow under the clock:**
1. **Recon** — `shape`, `info()`, `describe(include="all")`, `isna().sum().sort_values()`.
2. **Define target & drop leaks** — anything that encodes the answer (here: `alive`).
3. **Split FIRST** — `train_test_split(stratify=y)` before any fitting, so stats come from train only.
4. **Pipeline** — `ColumnTransformer(num: impute→scale, cat: impute→onehot)` → model. (Or just use it inside `cross_val_score` and it's leak-safe by construction.)
5. **Baseline → model → CV** — majority-class first, then the model, then `cross_val_score`.
6. **Evaluate & explain** — right metric (ROC-AUC/F1 for imbalanced, RMSE/MAE for regression) + feature importance.

**Snippets to type blind:**
```python
# group-wise impute
df[c] = df[c].fillna(df.groupby(g)[c].transform("median"))
# IQR fence
q1,q3 = s.quantile([.25,.75]); lo,hi = q1-1.5*(q3-q1), q3+1.5*(q3-q1)
# the pipeline (see Part 7)
# named groupby agg
df.groupby(k).agg(n=("x","size"), m=("x","mean"))
```

**Gotchas graders watch for:** fitting a scaler/imputer on full data (leakage) · forgetting `stratify` on imbalanced targets · accuracy on imbalanced data · one-hot without `handle_unknown="ignore"` (test-time crash) · leaving target-leak columns in.

**Now go do a Part-7 type-from-blank rep. That's tonight's commitment.**
